Запуск Qdrant в Docker с помощью Docker Compose:

```zsh
docker compose up -d
```

In [ ]:
#!pip install langchain-qdrant
#!pip install langchain_ollama
#!pip install langchain
#!pip install langchain_classic

# Загружаю PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(file_path="linux-manual.pdf")
docs = loader.load()

# Делю PDF на чанки

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
split_docs = text_splitter.split_documents(docs)

print(f'Кол-во чанков: {len(split_docs)}')

In [ ]:
len(split_docs)

# Записываем чанки и их эмбединги в йвкфте

In [ ]:
from langchain_ollama import OllamaEmbeddings, 
from langchain_qdrant import QdrantVectorStore

ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

In [ ]:
doc_store = QdrantVectorStore.from_documents(
    documents=split_docs,
    url="http://localhost:6333",
    collection_name="linux_manual",
    embedding=ollama_embeddings,
)

# Использование имеющейся коллекции

In [ ]:
doc_store = QdrantVectorStore.from_existing_collection(
    embedding=ollama_embeddings,
    collection_name="linux_manual",
    url="http://localhost:6333",
)

# Поиск документов в базе

In [ ]:
query = "How file permissions are altered?"
found_docs = doc_store.similarity_search(query, k=3)

In [ ]:
# found_docs[0]

# Запрос

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_core.prompts import PromptTemplate

template = """Given the following extracted parts of a long document and a question, create a final answer. 
If you don't know the answer, just say that you don't know. Don't try to make up an answer.
QUESTION: 
{question}
=========
CONTEXT:
{context}
=========
FINAL ANSWER:"""

PROMPT = PromptTemplate(template=template, input_variables=["context", "question"])

chain = load_qa_chain(
    OllamaLLM(model="qwen3:8b", temperature=0),
    chain_type="stuff",
    prompt=PROMPT,
)

query = "Как выдать права на файл?"
chain.run(input_documents=found_docs, question=query)